<a href="https://colab.research.google.com/github/alishabashir987-bot/alisha/blob/main/Assignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
"""Train a regression model and a classification model on the loan dataset.

Regression:     predict Loan_Amount (a number) with LinearRegression
Classification: predict Loan_Status (Approved / Rejected) with LogisticRegression

Both models are scikit-learn Pipelines (imputation + scaling + one-hot encoding),
so the backend can feed them raw form values directly. Saved with joblib.
"""
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Fixed: Use Path.cwd() instead of Path(__file__).parent in a notebook environment
BASE = Path.cwd()
# Fixed: Points to /content/train.csv directly based on available files
DATA = BASE / "train.csv"
MODELS = BASE / "models"
MODELS.mkdir(exist_ok=True)

NUMERIC = ["Dependents", "Applicant_Income", "Coapplicant_Income", "Loan_Term", "Credit_History", "Age"]
CATEGORICAL = ["Gender", "Married", "Education", "Employment_Status", "Property_Area"]


def make_preprocessor(numeric, categorical):
    return ColumnTransformer(
        [
            ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric),
            (
                "cat",
                Pipeline(
                    [
                        ("impute", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical,
            ),
        ]
    )


def main():
    df = pd.read_csv(DATA)

    # ---------------- Regression: predict Loan_Amount ----------------
    reg_df = df.dropna(subset=["Loan_Amount"])
    reg_features = NUMERIC + CATEGORICAL
    X_train, X_test, y_train, y_test = train_test_split(
        reg_df[reg_features], reg_df["Loan_Amount"], test_size=0.2, random_state=42
    )
    reg_model = Pipeline(
        [("prep", make_preprocessor(NUMERIC, CATEGORICAL)), ("model", LinearRegression())]
    )
    reg_model.fit(X_train, y_train)
    r2 = r2_score(y_test, reg_model.predict(X_test))
    print(f"Regression (Loan_Amount)  R2 on test set : {r2:.4f}")

    # ---------------- Classification: predict Loan_Status ----------------
    clf_features = NUMERIC + CATEGORICAL + ["Loan_Amount"]
    clf_numeric = NUMERIC + ["Loan_Amount"]
    X_train, X_test, y_train, y_test = train_test_split(
        df[clf_features], df["Loan_Status"], test_size=0.2, random_state=42, stratify=df["Loan_Status"]
    )
    clf_model = Pipeline(
        [
            ("prep", make_preprocessor(clf_numeric, CATEGORICAL)),
            ("model", LogisticRegression(max_iter=1000)),
        ]
    )
    clf_model.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf_model.predict(X_test))
    print(f"Classification (Loan_Status) accuracy     : {acc:.4f}")

    # ---------------- Save ----------------
    joblib.dump(reg_model, MODELS / "regression_model.joblib")
    joblib.dump(clf_model, MODELS / "classification_model.joblib")
    (MODELS / "metrics.txt").write_text(f"R2={r2:.4f}\nAccuracy={acc:.4f}\n")
    print(f"Saved models to {MODELS}")


if __name__ == "__main__":
    main()

Regression (Loan_Amount)  R2 on test set : -0.0054
Classification (Loan_Status) accuracy     : 0.9844
Saved models to /content/models


In [16]:
"""FastAPI backend: loads both saved models and exposes two POST endpoints."""
from pathlib import Path
from typing import Literal

import joblib
import pandas as pd
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel, Field

# Fixed: Use Path.cwd() instead of Path(__file__).parent in a notebook environment
BASE = Path.cwd()
regression_model = joblib.load(BASE / "models" / "regression_model.joblib")
classification_model = joblib.load(BASE / "models" / "classification_model.joblib")

app = FastAPI(title="Loan ML App")

# Allow the HTML page to call the API even if it is opened from a different origin (e.g. file://)
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


class LoanApplicant(BaseModel):
    Gender: Literal["Male", "Female"]
    Married: Literal["Yes", "No"]
    Dependents: int = Field(ge=0, le=3)
    Education: Literal["Graduate", "Not Graduate"]
    Employment_Status: Literal["Salaried", "Self-Employed", "Unemployed"]
    Applicant_Income: float = Field(ge=0)
    Coapplicant_Income: float = Field(ge=0)
    Loan_Term: int = Field(gt=0)
    Credit_History: Literal[0, 1]
    Property_Area: Literal["Urban", "Semiurban", "Rural"]
    Age: float = Field(ge=18, le=100)


class ClassificationInput(LoanApplicant):
    Loan_Amount: float = Field(gt=0)


@app.post("/predict/regression")
def predict_regression(data: LoanApplicant):
    """Predict the loan amount (a number) for an applicant."""
    row = pd.DataFrame([data.model_dump()])
    prediction = float(regression_model.predict(row)[0])
    return {"prediction": round(prediction, 2)}


@app.post("/predict/classification")
def predict_classification(data: ClassificationInput):
    """Predict whether the loan is Approved or Rejected."""
    row = pd.DataFrame([data.model_dump()])
    label = str(classification_model.predict(row)[0])
    probabilities = classification_model.predict_proba(row)[0]
    confidence = float(max(probabilities))
    return {"prediction": label, "confidence": round(confidence, 4)}


@app.get("/")
def index():
    return FileResponse(BASE / "frontend" / "index.html")